# 05 · Interpretación y validación de los clústeres

Caracteriza cada segmento en unidades originales y estandarizadas, revisa su distribución territorial y genera el dataset municipal con la etiqueta final.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 150)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = ROOT / 'data' / 'curated' / 'municipios.parquet'
MODELS_DIR = ROOT / 'outputs' / 'models'
REPORTS_DIR = ROOT / 'outputs' / 'reports'
FIGURES_DIR = ROOT / 'outputs' / 'figures'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

curated = pd.read_parquet(DATA_PATH)
modelo = pd.read_parquet(MODELS_DIR / 'dataset_modelo.parquet')
asignaciones = pd.read_parquet(MODELS_DIR / 'asignaciones_cluster.parquet')
X = np.load(MODELS_DIR / 'matriz_preprocesada.npy')
configuracion = json.loads((MODELS_DIR / 'configuracion_variables.json').read_text(encoding='utf-8'))
VARIABLES = configuracion['variables']

segmentado = modelo.merge(
    asignaciones[['codigo_ine', 'cluster']], on='codigo_ine', how='inner', validate='one_to_one'
)
print(segmentado.shape)

## Tamaño y perfil en unidades originales

In [ ]:
tamanios = segmentado.groupby('cluster').agg(
    municipios=('codigo_ine', 'size'),
    poblacion_total=('poblacion_2025', 'sum'),
    poblacion_mediana=('poblacion_2025', 'median'),
)
tamanios['municipios_pct'] = tamanios['municipios'].div(len(segmentado)).mul(100).round(2)
display(tamanios)

perfil_mediana = segmentado.groupby('cluster')[VARIABLES].median()
perfil_media = segmentado.groupby('cluster')[VARIABLES].mean()
perfil_mediana.to_csv(REPORTS_DIR / 'perfil_clusters_mediana.csv', encoding='utf-8')
perfil_media.to_csv(REPORTS_DIR / 'perfil_clusters_media.csv', encoding='utf-8')
display(perfil_mediana.T)

## Variables que más distinguen cada clúster

In [ ]:
estandarizado = pd.DataFrame(X, columns=VARIABLES)
estandarizado['cluster'] = asignaciones['cluster'].to_numpy()
perfil_estandarizado = estandarizado.groupby('cluster')[VARIABLES].mean()

diferenciadoras = (
    perfil_estandarizado.stack()
    .rename('valor_estandarizado')
    .reset_index()
    .assign(importancia=lambda x: x['valor_estandarizado'].abs())
    .sort_values(['cluster', 'importancia'], ascending=[True, False])
    .groupby('cluster', as_index=False)
    .head(7)
)
display(diferenciadoras)
diferenciadoras.to_csv(REPORTS_DIR / 'variables_diferenciadoras.csv', index=False, encoding='utf-8')

In [ ]:
fig, ax = plt.subplots(figsize=(max(12, len(VARIABLES) * 0.45), 5))
imagen = ax.imshow(perfil_estandarizado, aspect='auto', cmap='coolwarm', vmin=-2, vmax=2)
ax.set_xticks(range(len(VARIABLES)), VARIABLES, rotation=90)
ax.set_yticks(range(len(perfil_estandarizado)), perfil_estandarizado.index)
ax.set(xlabel='Variable', ylabel='Clúster', title='Perfil medio estandarizado')
fig.colorbar(imagen, ax=ax, label='Valor estandarizado medio')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'perfil_clusters_heatmap.png', dpi=180, bbox_inches='tight')
plt.show()

## Distribución provincial y mapa exploratorio

In [ ]:
distribucion_provincial = pd.crosstab(
    segmentado['provincia'], segmentado['cluster'], normalize='index'
).mul(100).round(1)
display(distribucion_provincial)
distribucion_provincial.to_csv(REPORTS_DIR / 'clusters_por_provincia_pct.csv', encoding='utf-8')

mapa = curated[['codigo_ine', 'longitud', 'latitud']].merge(
    asignaciones[['codigo_ine', 'cluster']], on='codigo_ine', how='inner', validate='one_to_one'
).dropna(subset=['longitud', 'latitud'])
fig, ax = plt.subplots(figsize=(10, 8))
dispersion = ax.scatter(
    mapa['longitud'], mapa['latitud'], c=mapa['cluster'], cmap='tab10', s=7, alpha=0.7
)
ax.set(title='Distribución espacial exploratoria de los clústeres', xlabel='Longitud', ylabel='Latitud')
fig.colorbar(dispersion, ax=ax, label='Clúster')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'clusters_mapa_exploratorio.png', dpi=180)
plt.show()

## Etiquetas interpretables

Rellena el diccionario tras analizar los perfiles. No nombres los clústeres únicamente por una variable.

In [ ]:
NOMBRES_CLUSTERS = {
    # 0: 'Municipios ...',
}

resultado_final = curated.merge(
    asignaciones[['codigo_ine', 'cluster']], on='codigo_ine', how='left', validate='one_to_one'
)
resultado_final['segmento'] = resultado_final['cluster'].map(NOMBRES_CLUSTERS)
resultado_final.to_parquet(REPORTS_DIR / 'municipios_segmentados.parquet', index=False)
resultado_final[['codigo_ine', 'municipio', 'provincia', 'cluster', 'segmento']].to_csv(
    REPORTS_DIR / 'municipios_segmentados.csv', index=False, encoding='utf-8-sig'
)
resultado_final[['codigo_ine', 'municipio', 'provincia', 'cluster', 'segmento']].head()

## Validaciones finales pendientes

- Repetir el ajuste con varios `random_state` y medir estabilidad.
- Comparar resultados con distintos umbrales de población y cuantiles de winsorización.
- Confirmar que los clústeres no sean únicamente regiones geográficas encubiertas.
- Describir los municipios sin etiqueta por quedar fuera del universo definido.